# CE541E08 — Unit 4 · Day 34 — pd.cut() and replace()
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 34 of 45 |
| **Topics** | pd.cut · IMD classification · replace · pd.crosstab |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 34"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Binning and Recoding

Two common data preparation tasks in hydrology:
- **Binning** — converting a continuous variable (flow in m³/s) into categories (Low, Medium, High) using `pd.cut()`
- **Recoding** — replacing codes or sentinel values with meaningful labels using `.replace()`

---
## Code Block 1 — pd.cut(): Flow Regime Classification

### What this code does

We classify 60 days of daily flow into 5 flow regime categories using `pd.cut()`, then compute the mean flow within each category.

### Why each step is taken

**`pd.cut(series, bins, labels)`:**
`bins` defines the breakpoints between categories as a list of boundary values. `labels` assigns a name to each interval. For bins `[0,100,300,600,1000,5000]` and 5 labels, there are 5 intervals: (0,100], (100,300], (300,600], (600,1000], (1000,5000].

**Why pd.cut instead of apply with if-elif:**
`pd.cut` is vectorised — it classifies all values at once in C code. An `apply` with if-elif calls Python 60 times. For large datasets, `pd.cut` is much faster.

**`groupby('Category', observed=True)`:**
`observed=True` suppresses a warning about unused category levels. Without it, Pandas would also show empty rows for categories with no data.

### Algorithm

```
1. Create 60-day daily flow Series with DatetimeIndex

2. pd.cut(series, bins=[0,100,300,600,1000,5000],
          labels=['Very Low','Low','Moderate','High','Very High'])
   → assigns a category label to each flow value

3. df.groupby('Category', observed=True)['Flow_m3s'].mean()
   → mean flow within each category
```

### Expected output

```
Flow value counts:
Category
Very Low      ...
Low           ...
Moderate      ...
High          ...
Very High     ...

Mean flow per category:
Category
Low        ...
Moderate   ...
...
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(2)
daily_flow = pd.Series(
    np.round(np.random.exponential(300, 60), 1),
    index=pd.date_range('2024-07-01', periods=60, freq='D'),
    name='Flow_m3s'
)

# pd.cut: bin continuous values into categories
# bins = breakpoints; labels = name for each interval
bins   = [0, 100, 300, 600, 1000, 5000]
labels = ['Very Low','Low','Moderate','High','Very High']
daily_flow_cat = pd.cut(daily_flow, bins=bins, labels=labels)

print("Flow value counts:")
print(daily_flow_cat.value_counts().sort_index())

# Combine into DataFrame for groupby
df = pd.DataFrame({'Flow_m3s':daily_flow, 'Category':daily_flow_cat})
print()
print("Mean flow per category:")
print(df.groupby('Category', observed=True)['Flow_m3s'].mean().round(1))

### 🔁 Try this

Change `bins=[0,100,300,600,1000,5000]` to `bins=[0,200,500,1000,5000]` with 4 labels.

- Does any category become empty?
- Which bin captures the most days?

---
## Code Block 2 — pd.cut(): IMD Rainfall Classification

### What this code does

We apply `pd.cut()` with the exact IMD thresholds to classify 122 days of monsoon rainfall, then print a summary table showing day count and percentage for each category.

### Why each step is taken

**`bins=[-0.1, 0, 15.5, 64.4, 115.5, 204.4, 1000]`:**
The first breakpoint is -0.1 (not 0) so that exactly 0.0 mm falls into the 'No rain' category. `pd.cut` uses left-open, right-closed intervals: (-0.1, 0] captures 0.0 mm exactly.

**`counts = categories.value_counts().sort_index()`:**
`sort_index()` sorts by category label alphabetically. Since labels are strings, this may not give the natural IMD order. In practice, you would define categories as `pd.CategoricalDtype` with an explicit order.

### Algorithm

```
1. 122-day monsoon rainfall Series

2. bins = IMD thresholds: [-0.1, 0, 15.5, 64.4, 115.5, 204.4, 1000]
   labels = 6 IMD category names

3. pd.cut(series, bins, labels)
   → Categorical Series with named bins

4. .value_counts() → count per category
   count/total*100 → percentage
```

### Expected output

```
IMD Rainfall Classification — Monsoon Season 2024
Category               Days      %
-------------------------------------
Extremely Heavy           1    0.8%
Heavy                     7    5.7%
Light                    33   27.0%
Moderate                 40   32.8%
No rain                  36   29.5%
Very Heavy                5    4.1%
TOTAL                   122
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(10)
daily_rain = pd.Series(
    np.round(np.random.exponential(20, 122), 1),
    index=pd.date_range('2024-06-01', periods=122, freq='D'),
    name='Rainfall_mm'
)

# IMD breakpoints — first bin starts at -0.1 so that 0.0 mm falls into 'No rain'
bins   = [-0.1, 0, 15.5, 64.4, 115.5, 204.4, 1000]
labels = ['No rain','Light','Moderate','Heavy','Very Heavy','Extremely Heavy']

categories = pd.cut(daily_rain, bins=bins, labels=labels)
counts = categories.value_counts().sort_index()
total  = len(daily_rain)

print("IMD Rainfall Classification — Monsoon Season 2024")
print(f"{'Category':<22} {'Days':>6} {'%':>7}")
print("-" * 37)
for cat, cnt in counts.items():
    print(f"{cat:<22} {cnt:>6} {cnt/total*100:>6.1f}%")
print(f"{'TOTAL':<22} {total:>6}")

### 🔁 Try this

Compute the **total rainfall in each IMD category**:

```python
df = pd.DataFrame({'Rain':daily_rain,'Cat':categories})
df.groupby('Cat',observed=True)['Rain'].sum().round(1)
```

Which category contributed the most rainfall to the monsoon total?

---
## Code Block 3 — replace(): Cleaning Sensor Data

### What this code does

We use `.replace()` to clean a DataFrame — replacing -999 missing values with NaN, and replacing short codes with full descriptions.

### Why each step is taken

**`df['Flow_m3s'].replace(-999.0, float('nan'))`:**
Replaces the missing-value sentinel (-999) with actual NaN. After this, `.mean()` and other statistics automatically exclude the NaN values (because Pandas skips NaN by default, unlike NumPy which requires `nanmean`).

**`df['QA_code'].replace({'G':'GOOD','M':'MISSING','H':'HIGH'})`:**
Dictionary form of replace — replaces multiple values at once. More readable than three separate calls to `.replace()`.

### Algorithm

```
1. df['Flow_m3s'].replace(-999.0, float('nan'))
   → -999 becomes NaN; .mean() will skip it

2. df['QA_code'].replace({'G':'GOOD','M':'MISSING','H':'HIGH'})
   → expands codes to full names

3. df['Soil_type'].replace({'SL':'Sandy Loam','CL':'Clay Loam','C':'Clay'})

4. df['Flow_m3s'].mean() → pandas skips NaN automatically
```

### Expected output

```
         Date  Flow_m3s  QA_Flag   Soil_name
0  2024-07-01     234.5     GOOD  Sandy Loam
1  2024-07-02     678.9     GOOD   Clay Loam
2  2024-07-03       NaN  MISSING        Clay
3  2024-07-04     890.2     HIGH  Sandy Loam
Valid flow mean: 601.20 m3/s
```

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Date'     : ['2024-07-01','2024-07-02','2024-07-03','2024-07-04'],
    'Flow_m3s' : [234.5, 678.9, -999.0, 890.2],
    'QA_code'  : ['G','G','M','H'],
    'Soil_type': ['SL','CL','C','SL'],
})

# Replace -999 sentinel with NaN — Pandas skips NaN in statistics automatically
df['Flow_m3s'] = df['Flow_m3s'].replace(-999.0, float('nan'))

# Dictionary replace: multiple old→new mappings in one call
df['QA_Flag']  = df['QA_code'].replace({'G':'GOOD','M':'MISSING','H':'HIGH'})
df['Soil_name']= df['Soil_type'].replace({'SL':'Sandy Loam','CL':'Clay Loam','C':'Clay'})

print(df[['Date','Flow_m3s','QA_Flag','Soil_name']])
# .mean() skips NaN automatically (unlike NumPy which needs nanmean)
print(f"Valid flow mean: {df['Flow_m3s'].mean():.2f} m3/s")

### 🔁 Try this

Add another row with `QA_code='X'` (unknown code) and run `.replace({'G':'GOOD','M':'MISSING','H':'HIGH'})`.

What value does the unrecognised code `'X'` get? How does this differ from `.map()`?

---
## Code Block 4 — pd.crosstab: Flow Regime vs Season

### What this code does

We create a cross-tabulation showing how many days fall into each combination of flow regime and season — useful for understanding whether monsoon season coincides with high-flow regime.

### Why each step is taken

**`pd.crosstab(rows, columns)`:**
Counts the number of occurrences for each combination of two categorical variables. Produces a matrix (contingency table) with one category on rows and the other on columns. This is a standard tool in exploratory data analysis.

**`.div(ct.sum()) * 100`:**
Normalises the counts to percentages. `.sum()` gives the column totals; dividing each column by its total gives the fraction. Multiply by 100 for percentages.

### Algorithm

```
1. Classify flow into regimes with pd.cut
2. Assign seasons with apply(lambda)

3. pd.crosstab(df['Regime'], df['Season'])
   → count of days in each regime×season combination

4. ct.div(ct.sum())*100 → percentage distribution
```

### Expected output

```
Flow regime vs Season (days count):
Season          Monsoon  Non-monsoon
Regime
High                ...          ...
Low                 ...          ...
Moderate            ...          ...
Very High           ...          ...
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(5)
dates = pd.date_range('2022-01-01','2024-12-31',freq='D')
base  = np.where((dates.month>=6)&(dates.month<=9), 400, 80)
flow  = np.round(np.maximum(base+np.random.normal(0,base*0.3,len(dates)),5),1)
df    = pd.DataFrame({'Flow_m3s':flow}, index=dates)
df['Month']  = df.index.month
df['Season'] = df['Month'].apply(lambda m:'Monsoon' if 6<=m<=9 else 'Non-monsoon')
df['Regime'] = pd.cut(df['Flow_m3s'],
                      bins=[0,100,300,600,5000],
                      labels=['Low','Moderate','High','Very High'])

# pd.crosstab: count occurrences of each Regime × Season combination
ct = pd.crosstab(df['Regime'], df['Season'])
print("Flow regime vs Season (days count):")
print(ct)
print()

# Normalise to percentages
pct = ct.div(ct.sum()) * 100
print("Percentage distribution:")
print(pct.round(1))

### 🔁 Try this

Add a third season: **Post-monsoon** (Oct-Nov).

Modify the lambda to return three seasons, then rerun the crosstab.

Which flow regime is most common in Post-monsoon?

---
## Session Summary — pd.cut() and replace()

| Method | Syntax | What it does |
|---|---|---|
| Bin into categories | `pd.cut(series, bins, labels)` | Assigns category label to each value |
| Count categories | `.value_counts().sort_index()` | Count per bin |
| Replace sentinel | `series.replace(-999, np.nan)` | Replace one value |
| Replace multiple | `series.replace({'A':'Alpha','B':'Beta'})` | Replace using dictionary |
| Cross-tabulation | `pd.crosstab(rows, cols)` | Count of each row×col combination |
| Normalise | `ct.div(ct.sum())*100` | Convert counts to percentages |
| NaN mean | `series.mean()` | Skips NaN automatically in Pandas |

---
## Day 34 Assignment

30-day paired rainfall and flow dataset:

1. Use `pd.cut()` to classify rainfall into IMD categories and flow into regime categories
2. Build a `pd.crosstab` showing how often each rainfall category coincides with each flow regime
3. Which combination is most frequent?

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np

np.random.seed(7)
n=30
df=pd.DataFrame({'Date':pd.date_range('2024-07-01',periods=n,freq='D'),
                 'Rainfall_mm':np.round(np.random.exponential(35,n),1),
                 'Flow_m3s':np.round(np.random.exponential(300,n),1)})

rain_bins=[-0.1,0,15.5,64.4,115.5,1000]
rain_labs=['No rain','Light','Moderate','Heavy','V.Heavy']
flow_bins=[0,100,400,800,9999]
flow_labs=['Low','Medium','High','Flood']

df['Rain_cat'] = pd.cut(df['Rainfall_mm'],bins=rain_bins,labels=rain_labs)
df['Flow_cat'] = pd.cut(df['Flow_m3s'],bins=flow_bins,labels=flow_labs)
print(df[['Date','Rainfall_mm','Rain_cat','Flow_m3s','Flow_cat']].head(10))
ct = pd.crosstab(df['Rain_cat'],df['Flow_cat'])
print("Crosstab:"); print(ct)

---
- [ ] Run all cells — verify outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit4_Pandas/CE541E08_U4_Day34.ipynb`
- [ ] Commit: `Day 34 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*